# Python reference 2: strings, formatting and regex

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-reference-02-strings-and-regex.ipynb)

String basics, every common `str` method (case, stripping, searching, splitting, replacing, testing, padding), f-strings and the format mini-language, templates, bytes and encodings, Unicode normalization, and the `re` module with groups, flags, lazy matching and lookarounds. Practise it in [Python practice 2: strings, formatting and regex](../practice/python-practice-02-strings-and-regex.ipynb).

*Last reviewed: November 2026, against Python 3.13.*

## Contents

| section | entries |
| --- | --- |
| [String basics](#string-basics) | [`str`: immutable text](#str-immutable-text) |
| [String methods](#string-methods) | [Case: `lower`, `upper`, `casefold`, `title`, `capitalize`, `swapcase`](#case-lower-upper-casefold-title-capitalize-swapcase), [Whitespace and affixes: `strip`, `lstrip`, `rstrip`, `removeprefix`, `removesuffix`](#whitespace-and-affixes-strip-lstrip-rstrip-removeprefix-removesuffix), [Searching: `find`, `index`, `count`, `startswith`, `endswith`](#searching-find-index-count-startswith-endswith), [Splitting and joining: `split`, `rsplit`, `splitlines`, `partition`, `join`](#splitting-and-joining-split-rsplit-splitlines-partition-join), [Replacing and mapping: `replace`, `translate`, `maketrans`](#replacing-and-mapping-replace-translate-maketrans), [Testing content: `isdigit`, `isdecimal`, `isnumeric`, `isalpha`, `isalnum`, `isspace`](#testing-content-isdigit-isdecimal-isnumeric-isalpha-isalnum-isspace), [Padding: `ljust`, `rjust`, `center`, `zfill`](#padding-ljust-rjust-center-zfill) |
| [Formatting](#formatting) | [f-strings and the format mini-language](#f-strings-and-the-format-mini-language), [`str.format` and `%` formatting](#strformat-and--formatting) |
| [Bytes, encodings and Unicode](#bytes-encodings-and-unicode) | [`encode`, `decode` and error handling](#encode-decode-and-error-handling), [Unicode normalization: `unicodedata.normalize`](#unicode-normalization-unicodedatanormalize) |
| [Regular expressions](#regular-expressions) | [`re` functions](#re-functions), [Groups, named groups and `Match` objects](#groups-named-groups-and-match-objects), [Flags, greedy vs lazy, and lookarounds](#flags-greedy-vs-lazy-and-lookarounds) |

## String basics

### `str`: immutable text

A `str` is an immutable sequence of Unicode code points. Indexing and slicing return new strings; methods never change
the original.

| operation | example | result |
| --- | --- | --- |
| index, negative index | `s[0]`, `s[-1]` | first, last character |
| slice `[start:stop:step]` | `s[1:4]`, `s[::-1]` | substring, reversed copy |
| membership | `"ell" in "hello"` | `True` |
| length | `len("héllo")` | 5 (characters, not bytes) |
| repeat, concatenate | `"-" * 3`, `a + b` | `"---"` |
| raw string | `r"\d+\n"` | backslashes kept: use for regex and Windows paths |
| multi-line | triple quotes (three `"` or three `'`) | keeps line breaks |

**Notes:** building a string with `+=` in a loop copies it each time; collect pieces in a list and `"".join(parts)`.

In [ ]:
s = "data engineering"
print(s[0], s[-1], s[5:16], s[::-1], "eng" in s, len("héllo"), len("héllo".encode()))
parts = [str(i) for i in range(5)]
print("-".join(parts), r"\d+\n", len(r"\n"), len("\n"))

## String methods

### Case: `lower`, `upper`, `casefold`, `title`, `capitalize`, `swapcase`

| method | result for `"straße Data"` |
| --- | --- |
| `lower()` | `straße data` |
| `upper()` | `STRASSE DATA` |
| `casefold()` | `strasse data`: aggressive lowercasing for **case-insensitive comparison** |
| `title()` | `Straße Data`: capitalizes after every non-letter (`"o'neil"` → `"O'Neil"`) |
| `capitalize()` | `Straße data` |

**Notes:** compare user-entered text with `casefold()`, not `lower()`.

In [ ]:
t = "straße Data"
print(t.lower(), "|", t.upper(), "|", t.casefold(), "|", t.title(), "|", t.capitalize(), "|", "o'neil mcdonald".title())
print("STRASSE".lower() == t.split()[0].lower(), "STRASSE".casefold() == t.split()[0].casefold())

### Whitespace and affixes: `strip`, `lstrip`, `rstrip`, `removeprefix`, `removesuffix`

`strip(chars=None)` removes **any combination** of the given characters from both ends (whitespace by default).
`removeprefix(p)` / `removesuffix(s)` (3.9+) remove one exact prefix or suffix.

**Notes:** `"report.csv".rstrip(".csv")` gives `"report"` only by luck: it strips the characters `.`, `c`, `s`, `v`, so
`"stats.csv".rstrip(".csv")` gives `"stat"`. Use `removesuffix`.

In [ ]:
print(repr("  x \t\n".strip()), repr("--x--".strip("-")), repr("xxhixx".lstrip("x")))
print("stats.csv".rstrip(".csv"), "|", "stats.csv".removesuffix(".csv"), "|", "tmp_orders".removeprefix("tmp_"))

### Searching: `find`, `index`, `count`, `startswith`, `endswith`

| method | returns | if not found |
| --- | --- | --- |
| `s.find(sub, start, end)` / `rfind` | lowest / highest index | `-1` |
| `s.index(sub)` / `rindex` | index | raises `ValueError` |
| `s.count(sub)` | non-overlapping occurrences | `0` |
| `s.startswith(prefix_or_tuple)`, `endswith` | bool | |

**Notes:** `startswith` and `endswith` accept a **tuple** of options: `name.endswith((".csv", ".tsv"))`.

In [ ]:
path = "s3://lake/raw/orders/2026/01/orders.csv"
print(path.find("/raw"), path.rfind("/"), path.count("/"), path.find("missing"), path.endswith((".csv", ".tsv")))

### Splitting and joining: `split`, `rsplit`, `splitlines`, `partition`, `join`

| method | example | result |
| --- | --- | --- |
| `split()` (no argument) | `" a  b ".split()` | `['a', 'b']`: splits on runs of whitespace, drops empties |
| `split(sep)` | `"a,,b".split(",")` | `['a', '', 'b']`: keeps empty fields |
| `split(sep, maxsplit)` / `rsplit(sep, maxsplit)` | `"a.b.c".rsplit(".", 1)` | `['a.b', 'c']` |
| `splitlines()` | `"a\nb\r\n"` | `['a', 'b']`: handles `\r\n` |
| `partition(sep)` / `rpartition` | `"k=v=w".partition("=")` | `('k', '=', 'v=w')`: always three parts |
| `sep.join(iterable)` | `",".join(["a", "b"])` | `'a,b'` (items must be strings) |

In [ ]:
print(" a  b ".split(), "a,,b".split(","), "a.b.c".rsplit(".", 1), "x\ny\r\n".splitlines())
key, _, value = "timeout=30=s".partition("=")
print(key, value, "|", "no_equals".partition("="))
print(", ".join(str(n) for n in [1, 2, 3]))

### Replacing and mapping: `replace`, `translate`, `maketrans`

`replace(old, new, count=-1)` swaps substrings. `str.maketrans(from, to, delete)` plus `translate` maps or deletes
**characters** in one pass, faster than chained `replace` calls.

In [ ]:
print("a-b-c".replace("-", "/"), "a-b-c".replace("-", "", 1))
table = str.maketrans({"(": None, ")": None, "-": None, " ": None})
print("(512) 555-0100".translate(table), "abc".translate(str.maketrans("abc", "xyz")))

### Testing content: `isdigit`, `isdecimal`, `isnumeric`, `isalpha`, `isalnum`, `isspace`

| method | true for | false for |
| --- | --- | --- |
| `isdecimal()` | `"123"` | `"²"`, `"-1"`, `"1.5"` |
| `isdigit()` | `"123"`, `"²"` | `"-1"`, `"1.5"` |
| `isnumeric()` | `"123"`, `"²"`, `"½"` | `"-1"` |
| `isalpha()`, `isalnum()` | letters; letters and digits (Unicode-aware) | `"a b"` |
| `isspace()` | `" \t\n"` | `""` |

**Notes:** none of them validate numbers; parse with `int()`/`Decimal()` in a `try`.

In [ ]:
for s in ["123", "²", "½", "-1", "1.5", "abc", "a1", " "]:
    print(f"{s!r:6} decimal={s.isdecimal()!s:5} digit={s.isdigit()!s:5} numeric={s.isnumeric()!s:5} alnum={s.isalnum()}")

### Padding: `ljust`, `rjust`, `center`, `zfill`

`ljust(width, fill=" ")`, `rjust`, `center` pad to a width; `zfill(width)` pads with zeros **after** a sign
(`"-7".zfill(3)` → `"-07"`). None of them truncate longer strings.

In [ ]:
print(repr("id".ljust(6, ".")), repr("42".rjust(6)), repr("x".center(5, "*")), "-7".zfill(4), "12345".zfill(3))

## Formatting

### f-strings and the format mini-language

`f"{expression!conversion:format_spec}"`. Conversions: `!r` (repr), `!s` (str), `!a` (ascii). `f"{x=}"` prints the
expression and its value (debugging). Format specs can be nested: `f"{value:{width}.{precision}f}"`.

Format spec: `[[fill]align][sign][#][0][width][grouping][.precision][type]`

| spec | input | output |
| --- | --- | --- |
| `>8`, `<8`, `^8`, `*^8` | `"ab"` | right, left, centre, centre with `*` fill |
| `,` / `_` | `1234567` | `1,234,567` / `1_234_567` |
| `.2f`, `,.2f` | `1234.5` | `1234.50`, `1,234.50` |
| `.1%` | `0.256` | `25.6%` |
| `+d`, ` d` | `5` | `+5`, ` 5` (sign always / space for positive) |
| `08.3f` | `3.14159` | `0003.142` |
| `.3e`, `.3g` | `12345.678` | `1.235e+04`, `1.23e+04` |
| `x`, `#x`, `b`, `o` | `255` | `ff`, `0xff`, `11111111`, `377` |
| `%Y-%m-%d` | a `date` | date formatting via `strftime` |

In [ ]:
from datetime import date

value, name, width = 1234.5, "orders", 12
print(f"{name!r} {value=} [{name:>10}] [{name:*^12}] {1234567:,} {value:,.2f} {0.256:.1%} {5:+d}")
print(f"[{3.14159:08.3f}] {12345.678:.3e} {255:#x} {255:b} {date(2026, 1, 5):%Y-%m-%d (%a)} [{value:{width}.1f}]")

### `str.format` and `%` formatting

`"{} {name}".format(a, name=b)` uses the same mini-language and is useful for templates stored as data (with
`format_map(dict)`). `"%s %d" % (a, n)` is the legacy style still used by `logging` messages. Prefer f-strings in code.
For user-supplied templates, `string.Template("$name").safe_substitute(...)` cannot execute expressions.

In [ ]:
from string import Template

template = "{table}: {rows:,} rows"
print(template.format(table="orders", rows=12000), template.format_map({"table": "items", "rows": 5}))
print("%s has %d rows" % ("orders", 12), Template("Hello $user, $missing").safe_substitute(user="ana"))

## Bytes, encodings and Unicode

### `encode`, `decode` and error handling

`str.encode(encoding="utf-8", errors="strict")` produces `bytes`; `bytes.decode(...)` goes back. Files and network data
are bytes; decode at the boundary.

| `errors=` | behaviour |
| --- | --- |
| `"strict"` (default) | raise `UnicodeDecodeError` / `UnicodeEncodeError` |
| `"replace"` | insert `�` (decode) or `?` (encode) |
| `"ignore"` | drop bad bytes (data loss) |
| `"backslashreplace"` | keep them as `\xNN` escapes (lossless and visible) |

**Notes:** `"utf-8-sig"` strips a byte-order mark that Excel puts at the start of CSV files; a stray BOM otherwise
appears as `﻿` in the first header.

In [ ]:
data = "café".encode("utf-8")
print(data, len(data), data.decode("utf-8"))
latin = "café".encode("latin-1")
print(latin.decode("utf-8", errors="replace"), latin.decode("utf-8", errors="backslashreplace"))
bom = "﻿id,name".encode("utf-8")
print(repr(bom.decode("utf-8")), repr(bom.decode("utf-8-sig")))

### Unicode normalization: `unicodedata.normalize`

The same visible text can have several code-point sequences (`é` as one character, or `e` plus a combining accent).
Normalize before comparing or using text as a key.

| form | effect |
| --- | --- |
| `NFC` | compose: one code point where possible (storage default) |
| `NFD` | decompose: base letter + combining marks |
| `NFKC` / `NFKD` | also fold compatibility forms (`ﬁ` → `fi`, full-width digits → ASCII digits) |

**Use it for:** dedup keys and joins on names; strip accents with `NFKD` and dropping combining marks.

In [ ]:
import unicodedata

a, b2 = "café", "café"
print(a == b2, unicodedata.normalize("NFC", a) == unicodedata.normalize("NFC", b2))
print(unicodedata.normalize("NFKC", "ﬁle １２３"))
no_accents = "".join(c for c in unicodedata.normalize("NFKD", "Crème Brûlée") if not unicodedata.combining(c))
print(no_accents)

## Regular expressions

### `re` functions

| function | returns |
| --- | --- |
| `re.match(p, s)` | a `Match` if the pattern matches at the **start**, else `None` |
| `re.fullmatch(p, s)` | a `Match` if the **whole** string matches: use for validation |
| `re.search(p, s)` | the first match anywhere |
| `re.findall(p, s)` | list of matched strings (or tuples of groups when the pattern has groups) |
| `re.finditer(p, s)` | iterator of `Match` objects (positions, named groups) |
| `re.sub(p, repl, s, count=0)` | string with replacements; `repl` can be a function of the match |
| `re.subn(...)` | `(new_string, number_of_replacements)` |
| `re.split(p, s, maxsplit=0)` | split on a pattern |
| `re.compile(p, flags)` | a reusable `Pattern` with the same methods |

**Notes:** always write patterns as raw strings (`r"\d+"`). Use `re.escape(text)` to match user text literally.

In [ ]:
import re

log = "2026-01-05 09:15:00 ERROR order=42 user=ana latency=120ms"
print(bool(re.match(r"\d{4}", log)), bool(re.fullmatch(r"\d{4}", log)), re.search(r"ERROR|WARN", log).group())
print(re.findall(r"(\w+)=(\w+)", log))
print(re.sub(r"\d", "#", "card 4242"), re.subn(r"\s+", " ", "a   b    c"), re.split(r"[;,]\s*", "a, b;c"))
print(re.sub(r"(\d+)ms", lambda m: f"{int(m.group(1)) / 1000}s", log))

### Groups, named groups and `Match` objects

`(...)` captures; `(?P<name>...)` captures by name; `(?:...)` groups without capturing. A `Match` gives `group(n)`,
`group("name")`, `groups()`, `groupdict()`, `start()`, `end()` and `span()`.

In [ ]:
pattern = re.compile(r"(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>[\d:]+) (?P<level>[A-Z]+)")
m = pattern.match(log)
print(m.group("level"), m.groups(), m.groupdict(), m.span("level"))
print([mm.group(0) for mm in re.finditer(r"(?:order|user)=\w+", log)])

### Flags, greedy vs lazy, and lookarounds

| feature | syntax | meaning |
| --- | --- | --- |
| ignore case | `re.I` / `(?i)` | case-insensitive |
| multi-line anchors | `re.M` | `^`/`$` match at every line |
| dot matches newline | `re.S` | `.` includes `\n` |
| verbose | `re.X` | whitespace and `#` comments allowed in the pattern |
| greedy / lazy | `.*` / `.*?` | longest / shortest match |
| lookahead / lookbehind | `(?=...)`, `(?!...)`, `(?<=...)`, `(?<!...)` | assert context without consuming it |
| backreference | `(\w)\1` | the same text again |

In [ ]:
html = "<b>bold</b> and <i>italic</i>"
print(re.findall(r"<.*>", html), re.findall(r"<.*?>", html))
print(re.findall(r"^\w+", "a 1\nb 2", re.M), re.findall(r"\d+(?=ms)", "120ms 30s 5ms"), re.findall(r"(?<=\$)\d+", "$5 and 7 and $10"))
amount = re.compile(r"""
    (?P<currency>[A-Z]{3})\s*   # ISO currency code
    (?P<value>\d+(?:\.\d{2})?)  # amount with optional cents
""", re.X)
print(amount.search("paid EUR 12.50 today").groupdict(), re.findall(r"(\w)\1", "bookkeeper"))

**Next:** practise these entries in [Python practice 2: strings, formatting and regex](../practice/python-practice-02-strings-and-regex.ipynb).